# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent) – A* Algorithm\n\n**Quy ước ma trận:**\n- `0` → Ô trống / Máy hút bụi\n- `1` → Bụi\n- `2` → Tường / Vật cản\n\n**Chiến lược A\*:**\n- Nếu ô hiện tại có bụi → hút ngay.\n- Dùng A\* (\u0192(n) = g(n) + h(n)) để tìm đường đi ngắn nhất đến ô bụi gần nhất.\n- Heuristic h(n) = khoảng cách Manhattan đến ô bụi đích.

In [ ]:
import numpy as np\nimport random\nimport heapq

In [ ]:
# ── Cấu hình ──\nROWS      = 5\nCOLS      = 7\nWALL_PROB = 0.15\nDUST_PROB = 0.35\nMAX_STEPS = 300\n\n# ── Tạo môi trường ngẫu nhiên ──\ndef create_env(rows, cols, wall_prob, dust_prob):\n    grid = np.zeros((rows, cols), dtype=int)\n    for r in range(rows):\n        for c in range(cols):\n            v = random.random()\n            if v < wall_prob:\n                grid[r][c] = 2\n            elif v < wall_prob + dust_prob:\n                grid[r][c] = 1\n    free = [(r, c) for r in range(rows) for c in range(cols) if grid[r][c] != 2]\n    pos = random.choice(free)\n    return grid, pos\n\ngrid, start = create_env(ROWS, COLS, WALL_PROB, DUST_PROB)\ntotal_dust = int(np.sum(grid == 1))\n\nprint(f\"Ma trận ban đầu (vị trí máy: {start}):\")\nprint(grid)\nprint(f\"Tổng bụi: {total_dust} ô\")

In [ ]:
# ── Agent (A* Algorithm) ──\nMOVES = {'UP': (-1,0), 'DOWN': (1,0), 'LEFT': (0,-1), 'RIGHT': (0,1)}\n\ndef manhattan(a, b):\n    return abs(a[0] - b[0]) + abs(a[1] - b[1])\n\ndef astar_find_nearest_dust(grid, start_pos):\n    \"\"\"\n    Dùng A* tìm đường đi ngắn nhất từ start_pos đến ô bụi gần nhất.\n    f(n) = g(n) + h(n)\n      g(n) = số bước thực tế từ start đến n\n      h(n) = heuristic Manhattan từ n đến ô bụi đích\n    Trả về (đường đi, ô bụi đích) hoặc (None, None) nếu không tìm thấy.\n    \"\"\"\n    rows, cols = grid.shape\n    sr, sc = start_pos\n    \n    # Tập các ô bụi còn lại\n    dust_cells = [(r, c) for r in range(rows) for c in range(cols) if grid[r, c] == 1]\n    if not dust_cells:\n        return None, None\n    \n    # A* tìm đường đến mọi ô bụi, chọn ô có f-cost thấp nhất\n    best_path = None\n    best_target = None\n    best_cost = float('inf')\n    \n    for target in dust_cells:\n        tr, tc = target\n        # A* search từ start_pos đến target\n        open_set = []\n        heapq.heappush(open_set, (manhattan(start_pos, target), 0, sr, sc))  # (f, g, r, c)\n        came_from = {}\n        g_score = {(sr, sc): 0}\n        closed = set()\n        found = False\n        \n        while open_set:\n            f, g, r, c = heapq.heappop(open_set)\n            \n            if (r, c) in closed:\n                continue\n            closed.add((r, c))\n            \n            if (r, c) == target:\n                found = True\n                break\n            \n            for d, (dr, dc) in MOVES.items():\n                nr, nc = r + dr, c + dc\n                if not (0 <= nr < rows and 0 <= nc < cols):\n                    continue\n                if grid[nr, nc] == 2:\n                    continue\n                if (nr, nc) in closed:\n                    continue\n                \n                new_g = g + 1\n                if (nr, nc) not in g_score or new_g < g_score[(nr, nc)]:\n                    g_score[(nr, nc)] = new_g\n                    h = manhattan((nr, nc), target)\n                    f_new = new_g + h\n                    heapq.heappush(open_set, (f_new, new_g, nr, nc))\n                    came_from[(nr, nc)] = (r, c, d)\n        \n        if found and g_score.get(target, float('inf')) < best_cost:\n            best_cost = g_score[target]\n            best_target = target\n            # Truy ngược để lấy đường đi\n            path = []\n            cur = target\n            while cur != (sr, sc):\n                pr, pc, direction = came_from[cur]\n                path.append((direction, MOVES[direction][0], MOVES[direction][1]))\n                cur = (pr, pc)\n            path.reverse()  # đường đi từ start → target\n            best_path = path\n    \n    return best_path, best_target\n\ndef run_agent(grid_in, start, max_steps):\n    grid = grid_in.copy()\n    rows, cols = grid.shape\n    pos = list(start)\n    history = set()\n    steps = 0\n    cleaned = 0\n    \n    while steps < max_steps:\n        r, c = pos\n        \n        # Nếu ô hiện tại có bụi → hút\n        if grid[r][c] == 1:\n            state = ((r, c), 'CLEAN')\n            if state in history:\n                return grid, steps, cleaned, 'THAT BAI', 'Lap lai hanh dong CLEAN tai ' + str((r, c))\n            history.add(state)\n            grid[r][c] = 0\n            cleaned += 1\n            steps += 1\n            if cleaned == total_dust:\n                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'\n            continue\n        \n        # A*: tìm đường đến ô bụi gần nhất\n        path, target = astar_find_nearest_dust(grid, pos)\n        if path is None or len(path) == 0:\n            if cleaned == total_dust:\n                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'\n            return grid, steps, cleaned, 'THAT BAI', 'Khong tim thay duong di den bui'\n        \n        # Lấy bước đi đầu tiên trong đường đi A*\n        direction, dr, dc = path[0]\n        state = ((r, c), direction)\n        if state in history:\n            return grid, steps, cleaned, 'THAT BAI', f'Lap lai hanh dong {direction} tai {(r, c)}'\n        history.add(state)\n        \n        pos = [r + dr, c + dc]\n        steps += 1\n    \n    return grid, steps, cleaned, 'THAT BAI', f'Vuot qua gioi han {max_steps} buoc'\n\n\nfinal_grid, steps, cleaned, status, reason = run_agent(grid, start, MAX_STEPS)\n\nprint(\"Ma tran sau khi chay:\")\nprint(final_grid)\nprint()\nprint(f\"So buoc di : {steps}\")\nprint(f\"Bui da hut : {cleaned} / {total_dust} o\")\nprint(f\"Trang thai : {status}\")\nprint(f\"Ly do      : {reason}\")